In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

os.makedirs('model', exist_ok=True)
print("Libraries loaded!")

Libraries loaded!


In [2]:
np.random.seed(42)
n = 3000

age         = np.random.randint(20, 80, n)
sex         = np.random.randint(0, 2, n)
bmi         = np.round(np.random.normal(27, 6, n).clip(15, 55), 1)
sbp         = np.random.randint(90, 200, n)
dbp         = np.random.randint(60, 130, n)
cholesterol = np.random.randint(140, 380, n)
glucose     = np.random.randint(70, 300, n)
smoking     = np.random.randint(0, 2, n)
diabetes    = np.random.randint(0, 2, n)
activity    = np.random.randint(0, 3, n)
alcohol     = np.random.randint(0, 2, n)
family_hx   = np.random.randint(0, 2, n)
bp_med      = np.random.randint(0, 2, n)

risk = (
    (age - 20) / 60 * 0.25
    + sex * 0.10
    + (bmi - 15) / 40 * 0.08
    + (sbp - 90) / 110 * 0.15
    + (cholesterol - 140) / 240 * 0.10
    + (glucose - 70) / 230 * 0.08
    + smoking * 0.12
    + diabetes * 0.10
    - (activity / 2) * 0.08
    + family_hx * 0.10
    + np.random.normal(0, 0.05, n)
)
cardio = (risk > 0.45).astype(int)

df = pd.DataFrame({
    'age': age, 'sex': sex, 'bmi': bmi,
    'systolic_bp': sbp, 'diastolic_bp': dbp,
    'cholesterol': cholesterol, 'glucose': glucose,
    'smoking': smoking, 'diabetes': diabetes,
    'activity': activity, 'alcohol': alcohol,
    'family_history': family_hx, 'bp_medication': bp_med,
    'cardio': cardio
})

print(df.shape)
print(df['cardio'].value_counts())
df.head()

(3000, 14)
cardio
1    1740
0    1260
Name: count, dtype: int64


,age,sex,bmi,systolic_bp,diastolic_bp,cholesterol,glucose,smoking,diabetes,activity,alcohol,family_history,bp_medication,cardio
0,58,0,26.2,98,103,367,84,0,1,0,1,1,1,1
1,71,1,32.0,151,117,356,235,0,0,0,1,1,1,1
2,48,0,24.4,183,103,367,220,1,0,2,0,0,0,1
3,34,0,17.4,110,93,298,249,1,1,0,0,0,0,1
4,62,1,37.5,165,84,170,263,1,1,0,0,1,1,1


In [3]:
X = df.drop('cardio', axis=1)
y = df['cardio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.8633
              precision    recall  f1-score   support

           0       0.89      0.78      0.83       259
           1       0.85      0.93      0.89       341

    accuracy                           0.86       600
   macro avg       0.87      0.85      0.86       600
weighted avg       0.87      0.86      0.86       600



In [4]:
joblib.dump(model,  'model/rf_model.pkl')
joblib.dump(scaler, 'model/scaler.pkl')
print("✅ Model saved to model/ folder!")

✅ Model saved to model/ folder!
